<a href="https://colab.research.google.com/github/DuhranDuhran/Internal_RAG_Agent/blob/main/RAG_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Retrieval Augmented Generation (RAG) Pipeline Refactor

## 1. Setup and Data Preparation

In [ ]:
!pip install -q google-genai chromadb

In [ ]:
import os

# Create a 'data' directory inside Colab environment
os.makedirs("data", exist_ok=True)

# Document 1: HR Policy
hr_policy = """ACME CORP HR POLICY: REMOTE WORK & STIPENDS (2026)

SECTION 1: HOME OFFICE STIPEND
- Full-time employees receive a $500 annual home-office equipment stipend.
- Receipts must be submitted through the Acme HR Portal prior to November 30.
- Equipment purchased using company stipends remains Acme Corp property.

SECTION 2: PTO & ACCRUAL
- Employees accrue 15 days of paid time off (PTO) per calendar year.
- A maximum of 5 unused PTO days can roll over into the following year.
- PTO requests exceeding 5 consecutive business days require 14 days advance manager approval.
"""

# Document 2: IT Security SOP
it_sop = """ACME CORP IT SOP: SECURITY & ACCESS CONTROL (2026)

SECTION 1: NETWORK & ACCESS SECURITY
- Passwords must be updated every 90 days and contain at least 16 characters.
- Company laptops must connect via AcmeVPN when accessing internal staging or production servers.

SECTION 2: INCIDENT REPORTING
- Report suspected phishing emails using the PhishAlarm Outlook add-in.
- Report critical data security incidents immediately via Slack channel #it-incidents or by emailing security@acme.com.
"""

# Save synthetic files into data/
with open("data/hr_policy.txt", "w") as f:
    f.write(hr_policy.strip())

with open("data/it_sop.txt", "w") as f:
    f.write(it_sop.strip())

print("✅ Created directory 'data/' containing hr_policy.txt and it_sop.txt")

✅ Created directory 'data/' containing hr_policy.txt and it_sop.txt


## 2. Indexing Documents with ChromaDB

In [ ]:
import os
import glob
import chromadb
import google.generativeai as genai
from google.colab import userdata

# 1. Initialize Gemini Client
api_key = userdata.get('Gemini_API_Key1')
genai.configure(api_key=api_key)

# 2. Parse and Chunk Local Documents
documents = []
metadatas = []
ids = []

for filepath in glob.glob("data/*.txt"):
    filename = os.path.basename(filepath)
    dept = "HR" if "hr" in filename else "IT"

    with open(filepath, "r") as f:
        content = f.read()

    # Split document by double line breaks (section level)
    sections = [sec.strip() for sec in content.split("\n\n") if sec.strip()]

    for idx, section in enumerate(sections):
        documents.append(section)
        metadatas.append({"source": filename, "department": dept})
        ids.append(f"{filename}_chunk_{idx}")

# 3. Generate Vector Embeddings
print("Generating vector embeddings...")
embeddings = []
for doc in documents:
    response = genai.embed_content(
        model="models/gemini-embedding-001", # Updated to use the correct available embedding model
        content=doc
    )
    embeddings.append(response['embedding'])

# 4. Store Chunks, Embeddings, and Metadata in ChromaDB
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="acme_internal_knowledge")

collection.add(
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas,
    ids=ids
)

print(f"✅ Successfully indexed {collection.count()} document chunks into ChromaDB!")

Generating vector embeddings...
✅ Successfully indexed 6 document chunks into ChromaDB!


## 3. Initialize Generative Model

In [ ]:
import google.generativeai as genai

# Initialize the Gemini Generative Model
gen_model = genai.GenerativeModel('gemini-3.6-flash') # Updated model name to gemini-3.6-flash

## 4. Define RAG Functions

In [ ]:
def semantic_search(query: str, k: int = 2):
    """Performs a semantic search against the ChromaDB collection using the modern embedding model.
    """
    # Generate embedding for the query using the available embedding model
    query_embedding = genai.embed_content(
        model="models/gemini-embedding-001", # Updated to use the correct available embedding model
        content=query
    )['embedding']

    # Query ChromaDB
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=k,
        include=['documents', 'metadatas', 'distances']
    )

    # Restructure results for easier use
    retrieved_documents = []
    if results['documents']:
        for i in range(len(results['documents'][0])):
            retrieved_documents.append({
                'document': results['documents'][0][i],
                'metadata': results['metadatas'][0][i],
                'distance': results['distances'][0][i]
            })
    return retrieved_documents

In [ ]:
def generate_answer(query: str, retrieved_documents: list):
    """Generates an answer using the gemini-3.6-flash model based on the query and retrieved documents,
    with strict grounding instructions.
    """
    # Construct prompt with retrieved context
    context = "\n---\n".join([doc['document'] for doc in retrieved_documents])

    # Use a strict grounding system instruction
    system_instruction = (
        "You are a helpful assistant. "
        "Use ONLY the provided information to answer the question. "
        "If the information does not contain the answer, "
        "you MUST clearly state 'I cannot provide an answer based on the provided context.' "
        "Do not use any outside knowledge." # Added strict grounding
    )

    prompt = f"""{system_instruction}

Question: {query}

Information:
{context}

Answer:"""

    try:
        response = gen_model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"An error occurred during generation: {e}"

## 5. Test the RAG System

In [ ]:
# Example Query 1: HR Policy
query1 = "How many PTO days can I accrue and what's the rollover policy?"
print(f"\nQuery 1: {query1}")
retrieved_docs1 = semantic_search(query1)
print("Retrieved Documents for Query 1:")
for doc in retrieved_docs1:
    print(f"  - Source: {doc['metadata']['source']}, Distance: {doc['distance']:.4f}\n    Content: {doc['document'][:100]}...")
answer1 = generate_answer(query1, retrieved_docs1)
print(f"\nAnswer 1:\n{answer1}")

# Example Query 2: IT Policy
query2 = "What should I do if I suspect a phishing email or a security incident?"
print(f"\nQuery 2: {query2}")
retrieved_docs2 = semantic_search(query2)
print("Retrieved Documents for Query 2:")
for doc in retrieved_docs2:
    print(f"  - Source: {doc['metadata']['source']}, Distance: {doc['distance']:.4f}\n    Content: {doc['document'][:100]}...")
answer2 = generate_answer(query2, retrieved_docs2)
print(f"\nAnswer 2:\n{answer2}")

# Example Query 3: Out of context
query3 = "What is the capital of France?"
print(f"\nQuery 3: {query3}")
retrieved_docs3 = semantic_search(query3)
print("Retrieved Documents for Query 3:")
for doc in retrieved_docs3:
    print(f"  - Source: {doc['metadata']['source']}, Distance: {doc['distance']:.4f}\n    Content: {doc['document'][:100]}...")
answer3 = generate_answer(query3, retrieved_docs3)
print(f"\nAnswer 3:\n{answer3}")


Query 1: How many PTO days can I accrue and what's the rollover policy?
Retrieved Documents for Query 1:
  - Source: hr_policy.txt, Distance: 0.3555
    Content: SECTION 2: PTO & ACCRUAL
- Employees accrue 15 days of paid time off (PTO) per calendar year.
- A ma...
  - Source: hr_policy.txt, Distance: 0.8157
    Content: SECTION 1: HOME OFFICE STIPEND
- Full-time employees receive a $500 annual home-office equipment sti...


ERROR:tornado.access:503 POST /v1beta/models/gemini-3.6-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 20531.61ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-3.6-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 29177.78ms



Answer 1:
Based on the provided information:

* **Accrual:** Employees accrue 15 days of paid time off (PTO) per calendar year.
* **Rollover Policy:** A maximum of 5 unused PTO days can roll over into the following year.

Query 2: What should I do if I suspect a phishing email or a security incident?
Retrieved Documents for Query 2:
  - Source: it_sop.txt, Distance: 0.4880
    Content: SECTION 2: INCIDENT REPORTING
- Report suspected phishing emails using the PhishAlarm Outlook add-in...
  - Source: it_sop.txt, Distance: 0.8566
    Content: SECTION 1: NETWORK & ACCESS SECURITY
- Passwords must be updated every 90 days and contain at least ...


ERROR:tornado.access:503 POST /v1beta/models/gemini-3.6-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 11639.49ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-3.6-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 20111.75ms



Answer 2:
Based on the provided context, here is what you should do:

* **For suspected phishing emails:** Report them using the PhishAlarm Outlook add-in.
* **For critical data security incidents:** Report them immediately via the Slack channel #it-incidents or by emailing security@acme.com.

Query 3: What is the capital of France?
Retrieved Documents for Query 3:
  - Source: it_sop.txt, Distance: 1.0221
    Content: SECTION 1: NETWORK & ACCESS SECURITY
- Passwords must be updated every 90 days and contain at least ...
  - Source: hr_policy.txt, Distance: 1.0496
    Content: SECTION 1: HOME OFFICE STIPEND
- Full-time employees receive a $500 annual home-office equipment sti...

Answer 3:
I cannot provide an answer based on the provided context.
